# 4 Classification

The linear regression model discussed in Chapter 3 assumes that the response
variable Y is quantitative. But in many situations, the response
variable is instead qualitative. For example, eye color is qualitative. 
Often qualitative variables are referred to as categorical; we will use these 
terms interchangeably. In this chapter, we study approaches for predicting
qualitative responses, a process that is known as classification. Predicting
a qualitative response for an observation can be referred to as *classifying* 
that observation, since it involves assigning the observation to a category,
or class. On the other hand, often the methods used for classification first
predict the probability that the observation belongs to each of the categories
of a qualitative variable, as the basis for making the classification.
In this sense they also behave like regression methods.

There are many possible classification techniques, or classifiers, that one might 
use to predict a qualitative response. We touched on some of these classifier
in Sections 2.1.5 and 2.2.3. In this chapter we discuss some widely-used
classifiers: logistic regression, linear discriminant analysis, quadratic discriminant 
analysis, naive Bayes, and K-nearest neighbors. The discussion of logistic regression 
is used as a jumping-off point for a discussion of generalized
linear models, and in particular, Poisson regression. We discuss
more computer-intensive classification methods in later chapters: these include
generalized additive models (Chapter 7); trees, random forests, and
boosting (Chapter 8); and support vector machines (Chapter 9).


## 4.1 An Overview of Classification

Classification problems occur often, perhaps even more so than regression
problems. Some examples include:

1. A person arrives at the emergency room with a set of symptoms
that could possibly be attributed to one of three medical conditions.
Which of the three conditions does the individual have?

2. An online banking service must be able to determine whether or not
a transaction being performed on the site is fraudulent, on the basis
of the user’s IP address, past transaction history, and so forth.


3. On the basis of DNA sequence data for a number of patients with
and without a given disease, a biologist would like to figure out which
DNA mutations are deleterious (disease-causing) and which are not.

Just as in the regression setting, in the classification setting we have a
set of training observations (x1, y1), . . . , (xn, yn) that we can use to build
a classifier. We want our classifier to perform well not only on the training
data, but also on test observations that were not used to train the classifier.
In this chapter, we will illustrate the concept of classification using the
simulated Default data set.

In this chapter, we will illustrate the concept of a classification using the simulated
*Default* data set. We are interested in predicting whether an
individual will default on his or her credit card payment, on the basis of
annual income and monthly credit card balance. The data set is displayed
in Figure 4.1.

<img src="images for notes/Chapter4_fig4_1.png" width="600">

In the left-hand panel of Figure 4.1, we have plotted annual
income and monthly credit card balance for a subset of 10,000 individuals.
The individuals who defaulted in a given month are shown in orange, and
those who did not in blue. (The overall default rate is about 3 %, so we
have plotted only a fraction of the individuals who did not default.) It
appears that individuals who defaulted tended to have higher credit card
balances than those who did not. In the center and right-hand panels of
Figure 4.1, two pairs of boxplots are shown. The first shows the distribution
of balance split by the binary default variable; the second is a similar plot
for income. In this chapter, we learn how to build a model to predict default
(Y ) for any given value of balance (X1) and income (X2). Since Y is not
quantitative, the simple linear regression model of Chapter 3 is not a good
choice: we will elaborate on this further in Section 4.2.

It is worth noting that Figure 4.1 displays a very pronounced relationship
between the predictor balance and the response default. In most real
applications, the relationship between the predictor and the response will
not be nearly so strong. However, for the sake of illustrating the classification
procedures discussed in this chapter, we use an example in which the
relationship between the predictor and the response is somewhat exaggerated.

### 4.2 Why Not Linear Regression?

We have stated that linear regression is not appropriate in the case of a
qualitative response. Why not?
Suppose that we are trying to predict the medical condition of a patient
in the emergency room on the basis of her symptoms. In this simplified
example, there are three possible diagnoses: stroke, drug overdose, and
epileptic seizure. We could consider encoding these values as a quantitative 
response variable, $Y$, as follows:

$$
Y = \begin{cases} 
1 & \text{if stroke;} \\
2 & \text{if drug overdose;} \\
3 & \text{if epileptic seizure.}
\end{cases}
$$

Using this coding, least squares could be used to fit a linear regression model 
to predict $Y$ on the basis of a set of predictors $X_1, \dots, X_p$. 
Unfortunately, this coding implies an ordering on the outcomes, putting 
drug overdose in between stroke and epileptic seizure, and insisting that 
the difference between stroke and drug overdose is the same as the difference between
drug overdose and epileptic seizure. In practice there is no particular
reason that this needs to be the case. For instance, one could choose an
equally reasonable coding,

$$
Y = \begin{cases} 
1 & \text{if epileptic seizure;} \\
2 & \text{if stroke;} \\
3 & \text{if drug overdose.}
\end{cases}
$$

which would imply a totally different relationship among the three conditions.
Each of these codings would produce fundamentally different linear
models that would ultimately lead to different sets of predictions on test
observations.

If the response variable’s values did take on a natural ordering, such as
*mild*, *moderate*, and *severe*, and we felt the gap between mild and moderate
was similar to the gap between moderate and severe, then a 1, 2, 3 coding
would be reasonable. Unfortunately, in general there is no natural way to
convert a qualitative response variable with more than two levels into a
quantitative response that is ready for linear regression.

For a binary (two level) qualitative response, the situation is better. 
For instance, perhaps there are only two possibilities for the patient’s medical 
condition: stroke and drug overdose. We could then potentially use the
dummy variable approach from Section 3.3.1 to code the response as follows:

$$
Y = \begin{cases} 
0 & \text{if stroke;} \\
1 & \text{if drug overdose.}
\end{cases}
$$

We could then fit a linear regression to this binary response, and predict
drug overdose if $\hat{Y} >0.5$ and stroke otherwise. In the binary case it is not
hard to show that even if we flip the above coding, linear regression will
produce the same final predictions.

For a binary response with a 0/1 coding as above, regression by least
squares is not completely unreasonable: it can be shown that the $X \hat{\beta}$ obtained
using linear regression is in fact an estimate of Pr(drug overdose|X)
in this special case. However, if we use linear regression, some of our estimates
might be outside the [0, 1] interval (see Figure 4.2), making them
hard to interpret as probabilities! Nevertheless, the predictions provide an
ordering and can be interpreted as crude probability estimates. Curiously,
it turns out that the classifications that we get if we use linear regression
to predict a binary response will be the same as for the linear discriminant
analysis (LDA) procedure we discuss in Section 4.4.

<img src="images for notes/Chapter4_fig4_2.png" width="600">

To summarize, there are at least two reasons not to perform classification
using a regression method: (a) a regression method cannot accommodate
a qualitative response with more than two classes; (b) a regression
method will not provide meaningful estimates of Pr(Y |X), even with just
two classes. Thus, it is preferable to use a classification method that is
truly suited for qualitative response values. In the next section, we present
logistic regression, which is well-suited for the case of a binary qualitative
response; in later sections we will cover classification methods that are
appropriate when the qualitative response has two or more classes.

## 4.3 Logistic Regression

Consider again the Default data set, where the response default falls into
one of two categories, Yes or No. Rather than modeling this response Y
directly, logistic regression models the probability that Y belongs to a particular
category.
For the Default data, logistic regression models the probability of default.
For example, the probability of default given balance can be written as

$\hspace{150pt} \mathbb{P} (default = Yes|balance).$

The values of Pr(default = Yes|balance), which we abbreviate p(balance),
will range between 0 and 1. Then for any given value of balance, a prediction
can be made for default. For example, one might predict default = Yes
for any individual for whom p(balance) > 0.5. Alternatively, if a company
wishes to be conservative in predicting individuals who are at risk for default,
then they may choose to use a lower threshold, such as p(balance) >
0.1.

### *4.3.1 The Logistic Model*

How should we model the relationship between p(X) = Pr(Y = 1|X) and
X? (For convenience we are using the generic 0/1 coding for the response.)
In Section 4.2 we considered using a linear regression model to represent
these probabilities:

$$ p(X) = \beta_0 + \beta_1 X \tag{4.1} $$

If we use this approach to predict default=Yes using balance, then we
obtain the model shown in the left-hand panel of Figure 4.2. Here we see
the problem with this approach: for balances close to zero we predict a
negative probability of default; if we were to predict for very large balances,
we would get values bigger than 1. These predictions are not sensible, since
of course the true probability of default, regardless of credit card balance,
must fall between 0 and 1. This problem is not unique to the credit default
data. Any time a straight line is fit to a binary response that is coded as
0 or 1, in principle we can always predict p(X) < 0 for some values of X
and p(X) > 1 for others (unless the range of X is limited).
To avoid this problem, we must model p(X) using a function that gives
outputs between 0 and 1 for all values of X. Many functions meet this
description. In logistic regression, we use the logistic function:

$$ p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}} \tag{4.2} $$

To fit the model (4.2), we use a method called maximum likelihood, which 
we discuss in the next section. The right-hand panel of Figure 4.2 illustrates
the fit of the logistic regression model to the Default data. Notice that for
low balances we now predict the probability of default as close to, but never
below, zero. Likewise, for high balances we predict a default probability
close to, but never above, one. The logistic function will always produce
an S-shaped curve of this form, and so regardless of the value of X, we
will obtain a sensible prediction. We also see that the logistic model is
better able to capture the range of probabilities than is the linear regression
model in the left-hand plot. The average fitted probability in both cases is
0.0333 (averaged over the training data), which is the same as the overall
proportion of defaulters in the data set.

After a bit of manipulation of (4.2), we find that

$$ \frac{p(X)}{1-p(X)} = e^{\beta_0 + \beta_1 X} \tag{4.3} $$

The quantity p(X)/[1−p(X)] is called the *odds*, and can take on any value
between odds 0 and $\infty $. Values of the odds close to 0 and $\infty$ indicate very low
and very high probabilities of default, respectively. For example, on average
1 in 5 people with an odds of 1/4 will default, since p(X) = 0.2 implies an
odds of $\frac{0.2}{1−0.2} = \frac{1}{4}$. Likewise, on average nine out of every ten people with
an odds of 9 will default, since p(x) = 0.9 implies an odds of $\frac{0.9}{1-0.9} = 9$. 
Odds are traditionally used instead of probabilities in horse-racing, since
they relate more naturally to the correct betting strategy.
By taking the logarithm of both sides of (4.3), we arrive at

$$ log \left( \frac{p(X)}{1-p(X)} \right) = \beta_0 + \beta_1 X \tag{4.4} $$

The left-hand side is called the log odds or logit. We see that the logistic
regression (4.2) has a logit that is linear in X.
Recall from Chapter 3 that in a linear regression model, $\beta_1$ gives the
average change in Y associated with a one-unit increase in X. By contrast,
in a logistic regression model, increasing X by one unit changes the log
odds by $\beta_1$ (4.4). Equivalently, it multiplies the odds by $e^{\beta_1}$ (4.3). However,
because the relationship between p(X) and X in (4.2) is not a straight line,
$\beta_1$ does not correspond to the change in p(X) associated with a one-unit
increase in X. The amount that p(X) changes due to a one-unit change in
X depends on the current value of X. But regardless of the value of X, if
$\beta_1$ is positive then increasing X will be associated with increasing p(X),
and if $\beta_1$ is negative then increasing X will be associated with decreasing
p(X). The fact that there is not a straight-line relationship between p(X)
and X, and the fact that the rate of change in p(X) per unit change in X
depends on the current value of X, can also be seen by inspection of the
right-hand panel of Figure 4.2.

### *4.3.2 Estimating the Regression Coefficients*

The coefficients $\beta_0$ and $\beta_1$ in (4.2) are unknown, and must be estimated
based on the available training data. In Chapter 3, we used the least squares
approach to estimate the unknown linear regression coefficients. Although
we could use (non-linear) least squares to fit the model (4.4), the more
general method of maximum likelihood is preferred, since it has better statistical
properties. The basic intuition behind using maximum likelihood to fit a 
logistic regression model is as follows: we seek estimates for $\beta_0$ and
$\beta_1$ such that the predicted probability $\hat{p}(x_i)$ of default for each individual,
using (4.2), corresponds as closely as possible to the individual’s observed
default status. In other words, we try to find $\beta_0$ and $\beta_1$ such that plugging
these estimates into the model for p(X), given in (4.2), yields a number
close to one for all individuals who defaulted, and a number close to zero
for all individuals who did not. This intuition can be formalized using a
mathematical equation called a *likelihood function*:

$$ \ell(\beta_0, \beta_1) = \prod_{i:y_i=1} p(x_i) \prod_{i':y_{i'}=0} (1 - p(x_{i'})) \tag{4.5} $$

The estimates $\hat{\beta_0}$ and $\hat{\beta_1}$ are chosen to *maximize* this likelihood function.
Maximum likelihood is a very general approach that is used to fit many
of the non-linear models that we examine throughout this book. In the
linear regression setting, the least squares approach is in fact a special case
of maximum likelihood. The mathematical details of maximum likelihood
are beyond the scope of this book. However, in general, logistic regression
and other models can be easily fit using statistical software such as R, and
so we do not need to concern ourselves with the details of the maximum
likelihood fitting procedure.

Table 4.1 shows the coefficient estimates and related information that
result from fitting a logistic regression model on the Default data in order
to predict the probability of default=Yes using balance. We see that $\hat{\beta_1}$ =
0.0055; this indicates that an increase in balance is associated with an
increase in the probability of default. To be precise, a one-unit increase in
balance is associated with an increase in the log odds of default by 0.0055
units.

Many aspects of the logistic regression output shown in Table 4.1 are
similar to the linear regression output of Chapter 3. For example, we can
measure the accuracy of the coefficient estimates by computing their standard
errors. The z-statistic in Table 4.1 plays the same role as the t-statistic
in the linear regression output, for example in Table 3.1 on page 77. For
instance, the z-statistic associated with $\beta_1$ is equal to $\hat{\beta_1}/SE(\hat{\beta_1})$, and so a
large (absolute) value of the z-statistic indicates evidence against the null
hypothesis $H_0 : \beta_1 = 0$. This null hypothesis implies that 

$p(X) = \frac{e^\beta_0}{1+e^{beta_0}}$ :

in other words, that the probability of default does not depend on balance.
Since the p-value associated with balance in Table 4.1 is tiny, we can reject
$H_0$. In other words, we conclude that there is indeed an association between
balance and probability of default. The estimated intercept in Table 4.1
is typically not of interest; its main purpose is to adjust the average fitted
probabilities to the proportion of ones in the data (in this case, the overall
default rate).

### *4.3.3 Making Predictions*

Once the coefficients have been estimated, we can compute the probability
of default for any given credit card balance. 

<img src="images for notes/Chapter4_tab4_1.png" width="600">

For example, using the coefficient
estimates given in Table 4.1, we predict that the default probability 
for an individual with a <span style="color: brown;">balance</span> of \$1,000 is

$$
\hat{p}(X) = \frac{e^{\hat{\beta}_0 + \hat{\beta}_1 X}}{1 + e^{\hat{\beta}_0 + \hat{\beta}_1 X}} = \frac{e^{-10.6513 + 0.0055 \times 1,000}}{1 + e^{-10.6513 + 0.0055 \times 1,000}} = 0.00576,
$$

which is below 1%. In contrast, the predicted probability of default for an
individual with a balance of $2, 000 is much higher, and equals 0.586 or
58.6 %.

One can use qualitative predictors with the logistic regression model using
the dummy variable approach from Section 3.3.1. As an example, the
Default data set contains the qualitative variable student. To fit a model
that uses student status as a predictor variable, we simply create a dummy
variable that takes on a value of 1 for students and 0 for non-students. The
logistic regression model that results from predicting probability of default
from student status can be seen in Table 4.2. 

<img src="images for notes/Chapter4_tab4_2.png" width="600">

The coefficient associated
with the dummy variable is positive, and the associated p-value is statistically significant. This indicates that students tend to have higher default probabilities than non-students:

$$
\begin{aligned}
\widehat{\text{Pr}}(\text{default}=\text{Yes} \mid \text{student}=\text{Yes}) &= \frac{e^{-3.5041 + 0.4049 \times 1}}{1 + e^{-3.5041 + 0.4049 \times 1}} = 0.0431, \\
\widehat{\text{Pr}}(\text{default}=\text{Yes} \mid \text{student}=\text{No}) &= \frac{e^{-3.5041 + 0.4049 \times 0}}{1 + e^{-3.5041 + 0.4049 \times 0}} = 0.0292.
\end{aligned}
$$


### *4.3.4 Multiple Logistic Regression*

We now consider the problem of predicting a binary response using multiple
predictors. By analogy with the extension from simple to multiple linear
regression in Chapter 3, we can generalize (4.4) as follows:

$$ log \left( \frac{p(X)}{1-p(X)} \right) = \beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p \tag{4.6} $$

where $X = (X_1, \ldots , X_p)$ are *p* predictors. Equation 4.6 can be rewritten as 

$$ p(X) = \frac{e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}}{1 + e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}} \tag{4.7} $$

Just as in Section 4.3.2, we use the maximum likelihood method to estimate
$\beta_0, \beta_1, \cdot , \beta_p$.

Table 4.3 shows the coefficient estimates for a logistic regression model
that uses balance, income (in thousands of dollars), and student status to
predict probability of default.

<img src="images for notes/Chapter4_tab4_3.png" width="600">

There is a surprising result here. The pvalues
associated with balance and the dummy variable for student status
are very small, indicating that each of these variables is associated with
the probability of default. However, the coefficient for the dummy variable
is negative, indicating that students are less likely to default than nonstudents.
In contrast, the coefficient for the dummy variable is positive in
Table 4.2. How is it possible for student status to be associated with an
increase in probability of default in Table 4.2 and a decrease in probability
of default in Table 4.3? 

The left-hand panel of Figure 4.3 provides a graphical illustration of this 
apparent paradox

<img src="images for notes/Chapter4_fig4_3.png" width="600">

The orange and blue solid lines
show the average default rates for students and non-students, respectively,
as a function of credit card balance. The negative coefficient for student in
the multiple logistic regression indicates that for a fixed value of balance
and income, a student is less likely to default than a non-student. Indeed,
we observe from the left-hand panel of Figure 4.3 that the student default
rate is at or below that of the non-student default rate for every value of
balance.

But the horizontal broken lines near the base of the plot, which
show the default rates for students and non-students averaged over all values
of balance and income, suggest the opposite effect: the overall student
default rate is higher than the non-student default rate. Consequently, there
is a positive coefficient for student in the single variable logistic regression
output shown in Table 4.2.

The right-hand panel of Figure 4.3 provides an explanation for this discrepancy.
The variables student and balance are correlated.

Students tend
to hold higher levels of debt, which is in turn associated with higher probability
of default. In other words, students are more likely to have large
credit card balances, which, as we know from the left-hand panel of Figure
4.3, tend to be associated with high default rates. Thus, even though
an individual student with a given credit card balance will tend to have a
lower probability of default than a non-student with the same credit card
balance, the fact that students on the whole tend to have higher credit card
balances means that overall, students tend to default at a higher rate than
non-students. This is an important distinction for a credit card company
that is trying to determine to whom they should offer credit. A student is
riskier than a non-student if no information about the student’s credit card
balance is available. However, that student is less risky than a non-student
*with the same credit card balance*!

This simple example illustrates the dangers and subtleties associated
with performing regressions involving only a single predictor when other
predictors may also be relevant. As in the linear regression setting, the
results obtained using one predictor may be quite different from those obtained
using multiple predictors, especially when there is correlation among
the predictors. In general, the phenomenon seen in Figure 4.3 is known as
confounding.

By substituting estimates for the regression coefficients from Table <span style="color: blue;">4.3</span> into <span style="color: blue;">(4.7)</span>, we can make predictions. For example, a student with a credit card balance of \$1,500 and an income of \$40,000 has an estimated probability of default of

$$
\hat{p}(X) = \frac{e^{-10.869 + 0.00574 \times 1,500 + 0.003 \times 40 - 0.6468 \times 1}}{1 + e^{-10.869 + 0.00574 \times 1,500 + 0.003 \times 40 - 0.6468 \times 1}} = 0.058. \tag{4.8}
$$

A non-student with the same balance and income has an estimated probability of default of

$$
\hat{p}(X) = \frac{e^{-10.869 + 0.00574 \times 1,500 + 0.003 \times 40 - 0.6468 \times 0}}{1 + e^{-10.869 + 0.00574 \times 1,500 + 0.003 \times 40 - 0.6468 \times 0}} = 0.105. \tag{4.9}
$$

(Here we multiply the <span style="color: brown;">income</span> coefficient estimate from Table <span style="color: blue;">4.3</span> by 40, rather than by 40,000, because in that table the model was fit with <span style="color: brown;">income</span> measured in units of \$1,000.)

### *4.3.5 Multinomial Logistic Regression*

We sometimes wish to classify a response variable that has more than two
classes. For example, in Section 4.2 we had three categories of medical condition
in the emergency room: stroke, drug overdose, epileptic seizure.
However, the logistic regression approach that we have seen in this section
only allows for K = 2 classes for the response variable.

It turns out that it is possible to extend the two-class logistic regression
approach to the setting of K > 2 classes. This extension is sometimes
known as multinomial logistic regression. To do this, we first select a single
class to serve as the baseline; without loss of generality, we select the Kth
class for this role. Then we replace the model (4.7) with the model

$$ \text{Pr}(Y = k | X = x) = \frac{e^{\beta_{k0} + \beta_{k1}x_1 + \dots + \beta_{kp}x_p}}{1 + \sum_{l=1}^{K-1} e^{\beta_{l0} + \beta_{l1}x_1 + \dots + \beta_{lp}x_p}} \tag{4.10} $$

for $k = 1, \dots, K-1$, and

$$ \text{Pr}(Y = K | X = x) = \frac{1}{1 + \sum_{l=1}^{K-1} e^{\beta_{l0} + \beta_{l1}x_1 + \dots + \beta_{lp}x_p}}. \tag{4.11} $$

It is not hard to show that for $k = 1, \dots, K-1$,

$$ \log \left( \frac{\text{Pr}(Y = k | X = x)}{\text{Pr}(Y = K | X = x)} \right) = \beta_{k0} + \beta_{k1}x_1 + \dots + \beta_{kp}x_p. \tag{4.12} $$

Notice that (4.12) is quite similar to (4.6). Equation 4.12 indicates that once
again, the log odds between any pair of classes is linear in the features.
It turns out that in (4.10)-(4.12), the decision to treat the *K*th class as the baseline is unimportant. For example, when classifying emergency room visits into *stroke*, *drug overdose*, and *epileptic seizure*, suppose that we fit two multinomial logistic regression models: one treating *stroke* as the baseline, another treating *drug overdose* as the baseline. The coefficient estimates will differ between the two fitted models due to the differing choice of baseline, but the fitted values (predictions), the log odds between any pair of classes, and the other key model outputs will remain the same. 

(Marco note, I checked and hyeap, its true I think).

Nonetheless, interpretation of the coefficients in a multinomial logistic
regression model must be done with care, since it is tied to the choice
of baseline. For example, if we set epileptic seizure to be the baseline,
then we can interpret $\beta_{stroke-j}$ as the log odds of stroke versus epileptic
seizure, given that x1 = · · · = xp = 0. Furthermore, a one-unit increase
in Xj is associated with a $\beta_{stroke-j}$ increase in the log odds of stroke over
epileptic seizure. Stated another way, if Xj increases by one unit, then

$$ \frac{\text{Pr}(Y = \text{stroke} | X = x)}{\text{Pr}(Y = \text{epileptic seizure} | X = x)} $$

increases by $e^{\beta_{STROKE j}}$

We now briefly present an alternative coding for multinomial logistic regression, known as the *softmax* coding. 

$$X^T X = \begin{bmatrix}
n & \sum x_{i1} & \sum x_{i2} \\
\sum x_{i1} & \sum x_{i1}^2 & \sum x_{i1}x_{i2} \\
\sum x_{i2} & \sum x_{i1}x_{i2} & \sum x_{i2}^2
\end{bmatrix}$$

$$(X^T X)^{-1} = \begin{bmatrix}
C_{00} & C_{01} & C_{02} \\
C_{10} & C_{11} & C_{12} \\
C_{20} & C_{21} & C_{22}
\end{bmatrix}$$

## 4.4 Generative Models for Classification

Logistic regression involves directly modeling $\mathbb{P}$(Y = k | X = x ) using the logistic function, given by (4.7). In statistical jargon, we model the conditional distribution of the response $Y$, given the predictor(s) $X$. We now consider an alternative and less direct approach to estimating these probabilities. In this new approach, we model the distribution of the rpedictors $X$ separately in each of the response classses (i.e. for each value of $Y$). We then use Bayes Theorem to flip these around into estimates for $\mathbb{P}$(Y=k | X=x). When the distribution of $X$ within each class is assumed to be normal, it turns out that the model is very similar in form to logistic regression. 

Why do we need another method, when we have logistic regression?

- When there is substantial separation between the two classes, the parameter estiamtes for the logistic regression model are surprisingly unstable. The methods we consider in this section do not suffer from this problem.
- If the distibution of the predictors $X$ is approximately normal in each of the classes and the sample size is small, then the approaches in this section may be more accurate than logistic regression.
- The methods in this section can be naturally extended to the case of more than two response classes. (In the case of more than two response classes, we can also use multinomial logistic regression from Section 4.3.5)

Suppose that we wish to classify an observation into one of K classes,
where K ( 2. In other words, the qualitative response variable Y can take
on K possible distinct and unordered values. Let $\pi_k$ represent the overall or *prior* probability that a randomly chosen observation comes from the *k*th class <span style="color:blue"> (Marco here, call this the equivalent of $\mathbb{P}[Y=k].)$</span> Let $f_k(X) = \mathbb{P}(X | Y = k)$ denote the *density function* of $X$ for an observation that comes from the *k*th class. In other words, $f_k(x)$ is relatively large if there is a high probability that an observation in the *k*th class has $X \approx x$, and $f_k(x)$ is small if it is very unlikely that an observation in the *k*th class has $X \approx x$. Then, *Bayes Theorem* states that:

$$ \mathbb{P} (Y = k | X = x ) = \frac{\pi_k f_k(x)}{\sum^K_{l=1} \pi_l f_l(x)} \tag{4.15}$$

<span style="color:blue"> (Marco here, the bottom comes from the fact that $\mathbb{P}(A \cap B) = \mathbb{P}(A | B) \mathbb{P}(A)$, and summing $\mathbb{P}(X \cap Y)$ over all values of Y just gives $\mathbb{P}(X)$</span>.

In accordance with our earlier notation, we will use the abbreviation $p_k(x)$ =
Pr(Y = k|X = x); this is the posterior probability that an observation X = x belongs to the kth class. That is, it is the probability that the
observation belongs to the kth class, given the predictor value for that
observation.

Equation 4.15 suggests that instead of directly computing the posterior
probability pk(x) as in Section 4.3.1, we can simply plug in estimates of $\pi_k$
and fk(x) into (4.15). In general, estimating $\pi_k$ is easy if we have a random
sample from the population: we simply compute the fraction of the training
observations that belong to the kth class. However, estimating the density
function fk(x) is much more challenging. As we will see, to estimate fk(x),
we will typically have to make some simplifying assumptions.

We know from Chapter 2 that the Bayes classifier, which classifies an
observation x to the class for which pk(x) is largest, has the lowest possible
error rate out of all classifiers. (Of course, this is only true if all of the
terms in (4.15) are correctly specified.) Therefore, if we can find a way to
estimate fk(x), then we can plug it into (4.15) in order to approximate the
Bayes classifier.

In the following sections, we discuss three classifiers that use different
estimates of fk(x) in (4.15) to approximate the Bayes classifier: linear discriminant
analysis, quadratic discriminant analysis, and naive Bayes.

##
4.4.1 Linear Discriminant Analysis for p = 1


For now, assume that p = 1 — that is, we have only one predictor. We would
like to obtain an estimate for fk(x) that we can plug into (4.15) in order to
estimate pk(x). We will then classify an observation to the class for which
pk(x) is greatest. To estimate fk(x), we will first make some assumptions
about its form.
In particular, we assume that fk(x) is normal or Gaussian. In the one
dimensional setting, the normal density takes the form

$$ f_k(x) = \frac{1}{\sqrt{2\pi\sigma_k}} exp\left( - \frac{1}{2\sigma^2_k}(x-\mu_k)^2\right) \tag{4.16}$$

where $\mu_k$ and $\sigma^2_k$ are the mean and variance parameters for the *k*th class. For now, lets assume further that $\sigma_1^2 = ... = \sigma_K^2$: that is, there is a shared variance term across all $K$ classes, which for simplicity we can denote by $\sigma^2$. Plugging (4.16) into (4.15) we find that

$$p_k(x) = \frac{\pi_k \frac{1}{\sqrt{2\pi}\sigma} \exp \left(-\frac{1}{2\sigma^2}(x - \mu_k)^2\right)}{\sum_{l=1}^K \pi_l \frac{1}{\sqrt{2\pi}\sigma} \exp \left(-\frac{1}{2\sigma^2}(x - \mu_l)^2\right)}. \tag{4.17}$$

(Note that in (4.17), $\pi_k$ denotes the prior probability that an observation
belongs to the kth class, not to be confused with $\pi =$ 3.14159, the mathematical
constant.) The Bayes classifier involves assigning an observation
X = x to the class for which (4.17) is largest. Taking the log of (4.17) and
rearranging the terms, it is not hard to show that this is equivalent to
assigning the observation to the class for which

$$\delta_k(x) = x \cdot \frac{\mu_k}{\sigma^2} - \frac{\mu_k^2}{2\sigma^2} + \log(\pi_k) \tag{4.18}$$

is largest. For instance, if $K = 2$ and $\pi_1 = \pi_2$, then the Bayes classifier assigns an observation to class 1 if $2x(\mu_1 - \mu_2) > \mu_1^2 - \mu_2^2$, and to class 2 otherwise. The Bayes decision boundary is the point for which $\delta_1(x) = \delta_2(x)$; one can show that this amounts to$$x = \frac{\mu_1^2 - \mu_2^2}{2(\mu_1 - \mu_2)} = \frac{\mu_1 + \mu_2}{2}. \tag{4.19}$$

An example is shown in the left-hand panel of Figure 4.4. The two normal density functions that are displayed, $f_1(x)$ and $f_2(x)$, represent two distinct classes. The mean and variance parameters for the two density functions are $\mu_1 = -1.25, \mu_2 = 1.25$, and $\sigma_1^2 = \sigma_2^2 = 1$. The two densities overlap, and so given that $X = x$, there is some uncertainty about the class to which the observations belong. 

<img src="images for notes/Chapter4_fig4_4.png" width="600">

If we assume that an observation is equally likely
to come from either class—that is, $\pi_1$ = $\pi_2$ = 0.5 — then by inspection of
(4.19), we see that the Bayes classifier assigns the observation to class 1
if x < 0 and class 2 otherwise. Note that in this case, we can compute
the Bayes classifier because we know that X is drawn from a Gaussian
distribution within each class, and we know all of the parameters involved.
In a real-life situation, we are not able to calculate the Bayes classifier.

In practice, even if we are quite certain of our assumption that X is
drawn from a Gaussian distribution within each class, to apply the Bayes
classifier we still have to estimate the parameters $\mu_1, ... \mu_K , \pi_1, ... \pi_K, $ and $\sigma^2$.
The *linear discriminant analysis* (LDA) method approximates the Bayes classifier by plugging estimates for $\pi_k$, $\mu_k$ and $\sigma^2$ into (4.18). In particular, the following estimates are used:

$$\hat{\mu}_k = \frac{1}{n_k} \sum_{i:y_i=k} x_i$$

$$\hat{\sigma}^2 = \frac{1}{n - K} \sum_{k=1}^K \sum_{i:y_i=k} (x_i - \hat{\mu}_k)^2 \tag{4.20}$$

where *n* is the total number of training observations, and $n_k$ is the number of training observations in the *k*th class. The estimate for $\mu_k$ is simply the average of all the training observations from the *k*th class, while $\hat{\sigma^2}$ can be seen as a weighted average of the sample variances for each of the *K* classes. Sometimes, we have knowledge of the class membership probabilites $\pi_1, ... \pi_K$ which can be used directly. In the absence of any additional information, LDA estimates $\pi_k$ using the proportion of the training observations that belong to the *k*th class. In other words,

$$\hat{\pi_k} = n_k / n \tag{4.21}$$

The LDA classifier plugs the estimates given in (4.20) and (4.21) into (4.18),
and assigns an observation X = x to the class for which

$$\hat{\delta}_k(x) = x \cdot \frac{\hat{\mu}_k}{\hat{\sigma}^2} - \frac{\hat{\mu}_k^2}{2\hat{\sigma}^2} + \log(\hat{\pi}_k) \tag{4.22}$$

is the largest. The word *linear* in the classifiers name stems from the fact that the *discriminant functions* $\hat{\delta_k}(x)$ in (4.22) are linear functions of *x* (as opposed to a more complex function of x).

The right-hand panel of Figure 4.4 displays a histogram of a random sample of 20 observations from each class. To implement LDA, we began by estimating $\pi_k$, $\mu_k$, and $\sigma^2$ using (4.20) and (4.21). We then computed the
decision boundary, shown as a black solid line, that results from assigning
an observation to the class for which (4.22) is largest. All points to the left
of this line will be assigned to the green class, while points to the right of
this line are assigned to the purple class. In this case, since n1 = n2 = 20,
we have $\hat{\pi_1} = \hat{\pi_2}$. As a result, the decision boundary corresponds to the midpoint between the sample means for the two classes, 
$(\hat{\mu_1} + \hat{\mu_2})/2$. The
figure indicates that the LDA decision boundary is slightly to the left of
the optimal Bayes decision boundary, which instead equals $(\mu_1 + \mu_2)/2 = 0$. How well does the LDA classifier perform on this data? Since this is simulated data, we can generate a large number of test observation in order to compute the Bayes error rate and the LDA test error rate. These are 10.6% and 11.1% respectively. In other words, the LDA classifier's error rate is only 0.5% above the smallest possible error rate! This indicates that LDA is performing pretty well on this data set. 

To reiterate, the LDA classifier results from assuming that the observations
within each class come from a normal distribution with a classspecific
mean and a common variance $\sigma^2$, and plugging in estimates for these parameters into the Bayes classifier. In Section 4.4.3, we will consider a less stringent set of assumptions, by allowing the observations in the kth class
to have a class-specific variance, $\sigma^2_k$.

## 4.4.2 Linear Discriminant Analysis for p>1

We now extend the LDA classifier to the case of multiple predictors. To
do this, we will assume that X = (X1,X2, . . . ,Xp) is drawn from a multivariate
Gaussian (or multivariate normal) distribution, with a class-specific
mean vector and a common covariance matrix.We begin with a brief review this distribution.

The multivariate Gaussian distribution assumes that each individual predictor
follows a one-dimensional normal distribution, as in (4.16), with some correlation between each pair of predictors. 

Two examples of multivariate
Gaussian distributions with p = 2 are shown in Figure 4.5. The height of
the surface at any particular point represents the probability that both X1
and X2 fall in a small region around that point. In either panel, if the surface
is cut along the X1 axis or along the X2 axis, the resulting cross-section
will have the shape of a one-dimensional normal distribution.

<img src="images for notes/Chapter4_fig4_5.png" width="600">

The left-hand panel of Figure 4.5 illustrates an example in which Var(X1) = Var(X2) and
Cor(X1,X2) = 0; this surface has a characteristic bell shape. However, the
bell shape will be distorted if the predictors are correlated or have unequal
variances, as is illustrated in the right-hand panel of Figure 4.5. In this
situation, the base of the bell will have an elliptical, rather than circular,
shape. To indicate that a p-dimensional random variable X has a multivariate
Gaussian distribution, we write $X ~ N/(\mu, \Sigma)$. Here E(X) = $\mu$ is the mean of X (a vector with *p* components), and Cov(X) = $\Sigma$ is the p*p covariance matrix of X. Formally, the multivariate Gaussian density is defined as 

$$f(x) = \frac{1}{(2\pi)^{p/2}|\Sigma|^{1/2}} \exp \left( -\frac{1}{2}(x - \mu)^T \Sigma^{-1} (x - \mu) \right). \tag{4.23}$$

In the case of *p*>1 predictors, the LDA classifier assumes that the observations in the *k*th class are drawn from a multivariate Gaussian distribution, N($\mu_k$, $\Sigma$), where $\mu_k$ is a class-specific mean vector, and $\Sigma$ is a covariance matrix that is common to all *K* classes. Plugging the density fucntion for the *k*th class, *f_k*(X=*x*), into (41.5) and performing a little bit of algebra reveals that the Bayes Classifier assigns an observation X = *x* to the class for which 

$$\delta_k(x) = x^T \Sigma^{-1} \mu_k - \frac{1}{2} \mu_k^T \Sigma^{-1} \mu_k + \log \pi_k \tag{4.24}$$

is the largest. This is the vector/matrix version of (4.18).

An example is shown in the left-hand panel of Figure 4.6.

<img src="images for notes/Chapter4_fig4_6.png" width="600">

Three equally sized Gaussian classes are shown with class-specific mean vectors and a
common covariance matrix. The three ellipses represent regions that contain
95 % of the probability for each of the three classes. The dashed lines
are the Bayes decision boundaries. In other words, they represent the set
of values x for which $\delta_k(x) = \delta_l(x)$; i.e. :

$$x^T \Sigma^{-1} \mu_k - \frac{1}{2} \mu_k^T \Sigma^{-1} \mu_k = x^T \Sigma^{-1} \mu_l - \frac{1}{2} \mu_l^T \Sigma^{-1} \mu_l \tag{4.25}$$

for $k \neq l$. (The log $\pi_k$ term from (4.24) has disappeared because each of the three classes has the same number of training observations; i.e. $\pi_k$ is the same for each class.) Note that there are three lines representing the
Bayes decision boundaries because there are three pairs of classes among
the three classes. That is, one Bayes decision boundary separates class 1
from class 2, one separates class 1 from class 3, and one separates class 2
from class 3. These three Bayes decision boundaries divide the predictor
space into three regions. The Bayes classifier will classify an observation
according to the region in which it is located.

Once again, we need to estimate the unknown parameters $\mu_1, \dots, \mu_K$, $\pi_1, \dots, \pi_K$, and $\Sigma$; the formulas are similar to those used in the one-dimensional case, given in (4.20). To assign a new observation $X = x$, LDA plugs these estimates into (4.24) to obtain quantities $\hat{\delta}_k(x)$, and classifies to the class for which $\hat{\delta}_k(x)$ is largest. Note that in (4.24) $\delta_k(x)$ is a linear function of $x$; that is, the LDA decision rule depends on $x$ only through a linear combination of elements. As previously deiscussed, this is the reason for the word *linear* in LDA.

In the right-hand panel of Figure 4.6, 20 observations drawn from each of
the three classes are displayed, and the resulting LDA decision boundaries
are shown as solid black lines. Overall, the LDA decision boundaries are
pretty close to the Bayes decision boundaries, shown again as dashed lines.
The test error rates for the Bayes and LDA classifiers are 0.0746 and 0.0770,
respectively. This indicates that LDA is performing well on this data.

We can perform LDA on the Default data in order to predict whether
or not an individual will default on the basis of credit card balance and
student status.4 The LDA model fit to the 10,000 training samples results
in a training error rate of 2.75 %. This sounds like a low error rate, but two
caveats must be noted.

• First of all, training error rates will usually be lower than test error
rates, which are the real quantity of interest. In other words, we
might expect this classifier to perform worse if we use it to predict
whether or not a new set of individuals will default. The reason is
that we specifically adjust the parameters of our model to do well on
the training data. The higher the ratio of parameters p to number
of samples n, the more we expect this overfitting to play a role. For
these data we don’t expect this to be a problem, since overfitting p = 2 and
n = 10,000.

• Second, since only 3.33 % of the individuals in the training sample
defaulted, a simple but useless classifier that always predicts that
an individual will not default, regardless of his or her credit card
balance and student status, will result in an error rate of 3.33 %. In
other words, the trivial null classifier will achieve an error rate that 
is only a bit higher than the LDA training set error rate.

In practice, a binary classifier such as this one can make two types of
errors: it can incorrectly assign an individual who defaults to the no default
category, or it can incorrectly assign an individual who does not default to
the default category. It is often of interest to determine which of these two
types of errors are being made. A confusion matrix, shown for the Default
data in Table 4.4, is a convenient way to display this information. The
table reveals that LDA predicted that a total of 104 people would default.

<img src="images for notes/Chapter4_tab4_4.png" width="600">

Of these people, 81 actually defaulted and 23 did not. Hence only 23 out
of 9,667 of the individuals who did not default were incorrectly labeled.
This looks like a pretty low error rate! However, of the 333 individuals who
defaulted, 252 (or 75.7 %) were missed by LDA. So while the overall error
rate is low, the error rate among individuals who defaulted is very high.
From the perspective of a credit card company that is trying to identify
high-risk individuals, an error rate of 252/333 = 75.7 % among individuals
who default may well be unacceptable.

Class-specific performance is also important in medicine and biology,
where the terms *sensitivity* and *specificity* characterize the performance of
a classifier or screening test. In this case, the sensitivity is the percentage of true
defaulters that are identified; it equals 24.3%. The spcificity is the percentage of
non-defaulters that are correctly identified; it equals (1-23/9667) = 99.8%.

Why does LDA do such a poor job of classifying the customers who default?
In other words, why does it have such low sensitivity? As we have
seen, LDA is trying to approximate the Bayes classifier, which has the lowest
total error rate out of all classifiers. That is, the Bayes classifier will
yield the smallest possible total number of misclassified observations, regardless
of the class from which the errors stem. Some misclassifications will
result from incorrectly assigning a customer who does not default to the
default class, and others will result from incorrectly assigning a customer
who defaults to the non-default class. In contrast, a credit card company
might particularly wish to avoid incorrectly classifying an individual who
will default, whereas incorrectly classifying an individual who will not default,
though still to be avoided, is less problematic. We will now see that it
is possible to modify LDA in order to develop a classifier that better meets
the credit card company’s needs.

The Bayes classifier works by assigning an observation to the class for
which the posterior probability pk(X) is greatest. In the two-class case, this
amounts to assigning an observation to the default class if

$$ \mathbb{P}(default = Yes | X = x) > 0.5 \tag{4.26}$$

Thus, the Bayes Classifier, and by extension LDA, uses a threshold of 50% for the posterior probability of default in order to assign an observation to the *default* class. However, if we are concerned about incorrectly predicting
the default status for individuals who default, then we can consider
lowering this threshold. For instance, we might label any customer with a
posterior probability of default above 20 % to the *default* class.
In other words, instead of assigning an observation to the *default* class if (4.26) holds, we could instead assign an observation to this class if

$$ \mathbb{P}(default = Yes | X = x) > 0.2 \tag{4.27}$$

The error rates that result from taking this approach are shown in Table 4.5.
Now LDA predicts that 430 individuals will default. Of the 333 individuals
who default, LDA correctly predicts all but 138, or 41.4 %. This is a vast
improvement over the error rate of 75.7 % that resulted from using the
threshold of 50 %.

<img src="images for notes/Chapter4_tab4_5.png" width="600">

However, this improvement comes at a cost: now 235
individuals who do not default are incorrectly classified. As a result, the
overall error rate has increased slightly to 3.73 %. But a credit card company
may consider this slight increase in the total error rate to be a small price to
pay for more accurate identification of individuals who do indeed default.

Figure 4.7 illustrates the trade-off that results from modifying the threshold
value for the posterior probability of default. Various error rates are
shown as a function of the threshold value. Using a threshold of 0.5, as in
(4.26), minimizes the overall error rate, shown as a black solid line. This
is to be expected, since the Bayes classifier uses a threshold of 0.5 and is
known to have the lowest overall error rate. But when a threshold of 0.5 is
used, the error rate among the individuals who default is quite high (blue
dashed line). As the threshold is reduced, the error rate among individuals
who default decreases steadily, but the error rate among the individuals
who do not default increases. How can we decide which threshold value is
best? Such a decision must be based on *domain knowledge*, such as detailed
information about the costs associated with default.

<img src="images for notes/Chapter4_fig4_7.png" width="600">

The ROC curve is a popular graphic for simultaneously displaying the two types of errors for all possible thresholds. The name “ROC” is historic,
and comes from communications theory. It is an acronym for receiver operating
characteristics. Figure 4.8 displays the ROC curve for the LDA
classifier on the training data. The overall performance of a classifier, summarized
over all possible thresholds, is given by the area under the (ROC)
curve (AUC).

<img src="images for notes/Chapter4_fig4_8.png" width="600">

An ideal ROC curve will hug the top left corner, so the larger
the AUC the better the classifier. For this data the AUC is 0.95, which is
close to the maximum of 1.0, so would be considered very good. We expect
a classifier that performs no better than chance to have an AUC of 0.5
(when evaluated on an independent test set not used in model training).
ROC curves are useful for comparing different classifiers, since they take
into account all possible thresholds. It turns out that the ROC curve for
the logistic regression model of Section 4.3.4 fit to these data is virtually
indistinguishable from this one for the LDA model, so we do not display it
here.

As we have seen above, varying the classifier threshold changes its true positive and false positive rate. These are also called the *sensitivity* and one minus the *specificity* of our classifier. Since there is an almost bewildering array of terms used in this context, we now give a summary. 
Table 4.6 shows the possible results when applying a classifier (or diagnostic test)
to a population. To make the connection with the epidemiology literature,
we think of “+” as the “disease” that we are trying to detect, and “−” as
the “non-disease” state. To make the connection to the classical hypothesis
testing literature, we think of “−” as the null hypothesis and “+” as the alternative (non-null) hypothesis. In the context of the Default data, “+”
indicates an individual who defaults, and “−” indicates one who does not.

<img src="images for notes/Chapter4_tab4_6.png" width="600">

<img src="images for notes/Chapter4_tab4_7.png" width="600">



## Quadratic Discriminant Analysis

As we have discussed, LDA assumes that the observations within each class
are drawn from a multivariate Gaussian distribution with a class-specific
mean vector and a covariance matrix that is common to all K classes.
Quadratic discriminant analysis (QDA) provides an alternative approach.

Like LDA, the QDA classifier results from assuming that the observations
from each class are drawn from a Gaussian distribution, and plugging estimates
for the parameters into Bayes’ theorem in order to perform prediction.
However, unlike LDA, QDA assumes that each class has its own
covariance matrix. That is, it assumes that an observation from the kth
class is of the form $X * N(\mu_k,\Sigma_k)$, where $\Sigma_k$ is a covariance matrix for
the kth class. Under this assumption, the Bayes classifier assigns an observation
X = x to the class for which

$$\begin{aligned}
\delta_k(x) &= -\frac{1}{2}(x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k) - \frac{1}{2} \log |\Sigma_k| + \log \pi_k \\
&= -\frac{1}{2}x^T \Sigma_k^{-1} x + x^T \Sigma_k^{-1} \mu_k - \frac{1}{2}\mu_k^T \Sigma_k^{-1} \mu_k - \frac{1}{2} \log |\Sigma_k| + \log \pi_k
\end{aligned} \tag{4.28}
$$


is largest. So the QDA classifier involves plugging estimates for !k, μk,
and &k into (4.28), and then assigning an observation X = x to the class
for which this quantity is largest. Unlike in (4.24), the quantity x appears
as a quadratic function in (4.28). This is where QDA gets its name.
Why does it matter whether or not we assume that the K classes share a
common covariance matrix? In other words, why would one prefer LDA to
QDA, or vice-versa? The answer lies in the bias-variance trade-off. When
there are p predictors, then estimating a covariance matrix requires estimating
p(p+1)/2 parameters. QDA estimates a separate covariance matrix
for each class, for a total of Kp(p+1)/2 parameters. With 50 predictors this
is some multiple of 1,275, which is a lot of parameters. By instead assuming
that the K classes share a common covariance matrix, the LDA model
becomes linear in x, which means there are Kp linear coefficients to estimate.
Consequently, LDA is a much less flexible classifier than QDA, and
so has substantially lower variance. This can potentially lead to improved
prediction performance. But there is a trade-off: if LDA’s assumption that
the K classes share a common covariance matrix is badly off, then LDA
can suffer from high bias. Roughly speaking, LDA tends to be a better bet
than QDA if there are relatively few training observations and so reducing
variance is crucial. In contrast, QDA is recommended if the training set is
very large, so that the variance of the classifier is not a major concern, or if
the assumption of a common covariance matrix for the K classes is clearly
untenable.

<img src="images for notes/Chapter4_fig4_9.png" width="600">

Figure 4.9 illustrates the performances of LDA and QDA in two scenarios.
In the left-hand panel, the two Gaussian classes have a common correlation
of 0.7 between $X_1$ and $X_2$. As a result, the Bayes decision boundary
is linear and is accurately approximated by the LDA decision boundary.
The QDA decision boundary is inferior, because it suffers from higher variance
without a corresponding decrease in bias. In contrast, the right-hand
panel displays a situation in which the orange class has a correlation of 0.7
between the variables and the blue class has a correlation of −0.7. Now
the Bayes decision boundary is quadratic, and so QDA more accurately
approximates this boundary than does LDA.

## 4.4.4 Naive Bayes

In previous sections, we used Bayes’ theorem (4.15) to develop the LDA
and QDA classifiers. Here, we use Bayes’ theorem to motivate the popular
naive Bayes classifier.

Recall that Bayes' Theorem (4.15) provides an expression for the posterior probability $p_k(x) = \mathbb{P}(Y=k | X = x)$ in terms of $\pi_1, ... \pi_K$ and $f_1(x), ... , f_K(x)$. As we saw in previous sections, estimating the prior probabilities $\pi_1, ... , \pi_K$ is typically straightforward: for instance, we can estimape $\hat{\pi_k}$ as the proportion of training observations belonging to the *k*th class, for *k* = 1, ... , K. 

However, estimating $f_1(x), \dots, f_K(x)$ is more subtle. Recall that $f_k(x)$ is the $p$-dimensional density function for an observation in the $k$th class, for $k = 1, \dots, K$. In general, estimating a $p$-dimensional density function is challenging. In LDA, we make a very strong assumption that greatly simplifies the task: we assume that $f_k$ is the density function for a multivariate normal random variable with class-specific mean $\mu_k$, and shared covariance matrix $\Sigma$. By contrast, in QDA, we assume that $f_k$ is the density function for a multivariate normal random variable with class-specific mean $\mu_k$, and class-specific covariance matrix $\Sigma_k$. By making these very strong assumptions, we are able to replace the very challenging problem of estimating $K$ $p$-dimensional density functions with the much simpler problem of estimating $K$ $p$-dimensional mean vectors and one (in the case of LDA) or $K$ (in the case of QDA) $(p \times p)$-dimensional covariance matrices.

The naive Bayes classifier takes a different tack for estimating $f_1(x)$ , ... , $f_K(x)$. Instead of assuming that these function belong to a particular family of distributions (e.g. multivariate normal), we instead make a single assumption:

*Within the kth class, the p predictors are independent*

States mathematically, this assumption means that for *k* = 1, ... , K, 

$$ f_k(x) = f_{k1}(x_1) \times f_{k2}(x_2) \times ... f_{kp}(x_p) \tag{4.29}$$

where $f_{kj}$ is the density function of the *j*th predictor among observations in the *k*th class.

Why is this assumption so powerful? Essentially, estimating a p-dimensional
density function is challenging because we must consider not only
the marginal distribution of each predictor — that is, the distribution 
of each predictor on its own — but also the joint distribution of the predictors
— that is, the association between the different predictors. In the case of
a multivariate normal distribution, the association between the different
predictors is summarized by the off-diagonal elements of the covariance
matrix. However, in general, this association can be very hard to characterize,
and exceedingly challenging to estimate. But by assuming that the
p covariates are independent within each class, we completely eliminate the
need to worry about the association between the p predictors, because we
have simply assumed that there is no association between the predictors!

Do we really believe the naive Bayes assumption that the p covariates
are independent within each class? In most settings, we do not. But even
though this modeling assumption is made for convenience, it often leads to
pretty decent results, especially in settings where n is not large enough relative
to p for us to effectively estimate the joint distribution of the predictors
within each class. In fact, since estimating a joint distribution requires such
a huge amount of data, naive Bayes is a good choice in a wide range of settings.
Essentially, the naive Bayes assumption introduces some bias, but
reduces variance, leading to a classifier that works quite well in practice as
a result of the bias-variance trade-off.

Once we have made the naive Bayes assumption, we can plug (4.29) into
(4.15) to obtain an expression for the posterior probability,

$$\text{Pr}(Y = k|X = x) = \frac{\pi_k \times f_{k1}(x_1) \times f_{k2}(x_2) \times \dots \times f_{kp}(x_p)}{\sum_{l=1}^K \pi_l \times f_{l1}(x_1) \times f_{l2}(x_2) \times \dots \times f_{lp}(x_p)} \tag{4.30}$$

for $k = 1, \dots, K$. 

To estimate the one-dimensional density function $f_{kj}$ using training data $x_{1j}, \dots, x_{nj}$, we have a few options.

• If Xj is quantitative, then we can assume that Xj |Y = k ~ $N(\mu_{jk}, \sigma^2_{jk})$
jk). In other words, we assume that within each class, the jth predictor is
drawn from a (univariate) normal distribution. While this may sound
a bit like QDA, there is one key difference, in that here we are assuming
that the predictors are independent; this amounts to QDA with
an additional assumption that the class-specific covariance matrix is
diagonal.

• If Xj is quantitative, then another option is to use a non-parametric
estimate for fkj. A very simple way to do this is by making a histogram
for the observations of the jth predictor within each class.
Then we can estimate $f_{kj}(x_j)$ as the fraction of the training observations
in the kth class that belong to the same histogram bin as
xj . Alternatively, we can use a kernel density estimator, which is

• If Xj is qualitative, then we can simply count the proportion of training
observations for the jth predictor corresponding to each class. For
instance, suppose that Xj " {1, 2, 3}, and we have 100 observations
in the kth class. Suppose that the jth predictor takes on values of 1,
2, and 3 in 32, 55, and 13 of those observations, respectively. Then
we can estimate fkj as

$$\hat{f}_{kj}(x_j) = \begin{cases} 
0.32 & \text{if } x_j = 1 \\ 
0.55 & \text{if } x_j = 2 \\ 
0.13 & \text{if } x_j = 3. 
\end{cases}$$

We now consider the naive Bayes classifier in a toy example with $p = 3$ predictors and $K = 2$ classes. The first two predictors are quantitative, and the third predictor is qualitative with three levels. Suppose further that $\hat{\pi}_1 = \hat{\pi}_2 = 0.5$. The estimated density functions $\hat{f}_{kj}$ for $k = 1,2$ and $j = 1,2,3$ are displayed in Figure 4.10. 

<img src="images for notes/Chapter4_fig4_9.png" width="600">

Now suppose that we wish to classify a new observation, $x^* = (0.4, 1.5, 1)^T$. It turns out that in this example, $\hat{f_{12}}(0.4) = 0.368$, $\hat{f_{12}}(1.5) = 0.484$, $\hat{f_{13}}(1) = 0.226$, and $\hat{f_{21}}(0.4) =
0.030$, $\hat{f_{22}}(1.5) = 0.130$, $\hat{f_{23}}(1) = 0.616$. Plugging these estimates into (4.30)
results in posterior probability estimates of Pr(Y = 1|X = x&) = 0.944 and
$\mathbb{P}(Y = 2|X = x*) = 0.056$.

Table 4.8 provides the confusion matrix resulting from applying the naive
Bayes classifier to the Default data set, where we predict a default if the
posterior probability of a default — that is, P(Y = default|X = x) — exceeds
0.5.

<img src="images for notes/Chapter4_tab4_8.png" width="600">

Comparing this to the results for LDA in Table 4.4, our findings
are mixed. 

<img src="images for notes/Chapter4_tab4_4.png" width="600">

While LDA has a slightly lower overall error rate, naive Bayes
correctly predicts a higher fraction of the true defaulters. In this implementation
of naive Bayes, we have assumed that each quantitative predictor is
drawn from a Gaussian distribution (and, of course, that within each class,
each predictor is independent).

Just as with LDA, we can easily adjust the probability threshold for
predicting a default. For example, Table 4.9 provides the confusion matrix
resulting from predicting a default if P(Y = default|X = x) > 0.2. Again,
the results are mixed relative to LDA with the same threshold (Table 4.5).
Naive Bayes has a higher error rate, but correctly predicts almost two-thirds
of the true defaults.

In this example, it should not be too surprising that naive Bayes does
not convincingly outperform LDA: this data set has n = 10,000 and p = 2,
and so the reduction in variance resulting from the naive Bayes assumption
is not necessarily worthwhile. We expect to see a greater pay-off to using
naive Bayes relative to LDA or QDA in instances where p is larger or n is
smaller, so that reducing the variance is very important.

## 4.5 A comparison of Classification Models

### 4.5.1 An Analytical Comparison

We now perform an analytical (or mathematical) comparison of LDA, QDA,
naive Bayes, and logistic regression. We consider these approaches in a
setting with K classes, so that we assign an observation to the class that
maximizes Pr(Y = k|X = x). Equivalently, we can set K as the baseline
class and assign an observation to the class that maximizes-dsaokdjad

Ok Marco here. Basically they do some algebraic manipulation of the Log Odds, and get these results:

<img src="images for notes/Chapter4_fig4_90.png" width="600">



### 4.5.2 An Empirical Comparison.

We now compare the *empirical* (practical) performance of logistic regression, LDA, QDA, naive bayes, and KNN.

## 4.6 Generalized Linear Models

In Chapter 3, we assumed that the response Y is quantitative, and explored the use of least squares linear regression to predict $Y$. Thus far in this chapter, we have instead assumed that $Y$ is qualitative. However, we may sometimes be faced with situations in which Y is neither qualitative
nor quantitative, and so neither linear regression from Chapter 3 nor the
classification approaches covered in this chapter is applicable.

As a concrete example, we consider the Bikeshare data set. The response
is bikers, the number of hourly users of a bike sharing program in Washington,
DC. This response value is neither qualitative nor quantitative:
instead, it takes on non-negative integer values, or counts.

We will consider predicting bikers using the covariates mnth (month of the year), hr (hour
of the day, from 0 to 23), workingday (an indicator variable that equals 1 if
it is neither a weekend nor a holiday), temp (the normalized temperature,
in Celsius), and weathersit (a qualitative variable that takes on one of four
possible values: clear; misty or cloudy; light rain or light snow; or heavy
rain or heavy snow.)
In the analyses that follow, we will treat mnth, hr, and weathersit as
qualitative variables.

### 4.6.1 Linear Regression on the bikeshare data

To begin, we consider predicting bikers using linear regression. The results are shown in Table 4.10.

<img src="images for notes/Chapter4_tab4_10.png" width="600">

We see, for example, that a progression of weather from clear to cloudy
results in, on average, 12.89 fewer bikers per hour; however, if the weather
progresses further to rain or snow, then this further results in 53.60 fewer
bikers per hour. Figure 4.13 displays the coefficients associated with mnth

<img src="images for notes/Chapter4_fig4_13.png" width="600">

and the coefficients associated with hr. We see that bike usage is highest in
the spring and fall, and lowest during the winter months. Furthermore, bike
usage is greatest around rush hour (9 AM and 6 PM), and lowest overnight.
Thus, at first glance, fitting a linear regression model to the Bikeshare data
set seems to provide reasonable and intuitive results.

But upon more careful inspection, some issues become apparent. For
example, 9.6% of the fitted values in the Bikeshare data set are negative:
that is, the linear regression model predicts a negative number of users
during 9.6% of the hours in the data set. This calls into question our ability
to perform meaningful predictions on the data, and it also raises concerns
about the accuracy of the coefficient estimates, confidence intervals, and
other outputs of the regression model.

Furthermore, it is reasonable to suspect that when the expected value
of bikers is small, the variance of bikers should be small as well. For
instance, at 2 AM during a heavy December snow storm, we expect that
extremely few people will use a bike, and moreover that there should be
little variance associated with the number of users during those conditions.
This is borne out in the data: between 1 AM and 4 AM, in December,
January, and February, when it is raining, there are 5.05 users, on average,
with a standard deviation of 3.73. By contrast, between 7 AM and 10 AM,
in April, May, and June, when skies are clear, there are 243.59 users, on
average, with a standard deviation of 131.7. The mean-variance relationship
is displayed in the left-hand panel of Figure 4.14. 

<img src="images for notes/Chapter4_fig4_13.png" width="600">

This is a major violation
of the assumptions of a linear model, which state that 
$Y = \sum_{j=1}^p X_j \beta_j + \epsilon$
where $\epsilon$ is a mean-zero error term with variance $\sigma^2$ that is *constant*, and
not a function of the covariates. Therefore, the heteroscedasticity of the
data calls into question the suitability of a linear regression model.

Finally, the response bikers is integer-valued. But under a linear model,

$Y = \beta_0 + \sum_{j=1}^p X_j \beta_j + \epsilon$

where $\epsilon$ is a continuous-valued error term. This
means that in a linear model, the response $Y$ is necessarily continuousvalued
(quantitative). Thus, the integer nature of the response bikers suggests
that a linear regression model is not entirely satisfactory for this data set.

Some of the problems that arise when fitting a linear regression model
to the Bikeshare data can be overcome by transforming the response; for
instance, we can fit the model

$$log(Y) = \sum^p_{j=1} X_j \beta_j + \epsilon $$

Transforming the response avoids the possibility of negative predictions,
and it overcomes much of the heteroscedasticity in the untransformed data,
as is shown in the right-hand panel of Figure 4.14. However, it is not quite
a satisfactory solution, since predictions and inference are made in terms of
the log of the response, rather than the response. This leads to challenges
in interpretation, e.g. *“a one-unit increase in $X_j$ is associated with an
increase in the mean of the log of $Y$ by an amount $\beta_j$ ”.*

Furthermore, a log transformation of the response cannot be applied in settings where the
response can take on a value of 0. Thus, while fitting a linear model to
a transformation of the response may be an adequate approach for some
count-valued data sets, it often leaves something to be desired. We will see
in the next section that a Poisson regression model provides a much more
natural and elegant approach for this task.

#### 4.6.2 Poisson Regression on the Bikeshare Data

To overcome the inadequacies of linear regression for analyzing the Bikeshare data set, we will make use of an alternative approach,
called the *Poisson regression*. Before we can talk about Poisson regression, we must first introduce the Poisson distribution.
Suppose that a random variable Y takes on nonnegative integer values,
i.e. $Y \in \{0, 1, 2, . . .\}$. If Y follows the Poisson distribution, then

$$ \mathbb{P}(Y=k) = \frac{e^{-\lambda} \lambda^k}{k!} \hspace{2pt} for \hspace{3pt} k = 0,1,2,... \tag{4.35}$$

Here, $\lambda$ > 0 is the expected value of Y , i.e. E(Y ). It turns out that ! also
equals the variance of Y , i.e. $\lambda$ = E(Y ) = Var(Y ). This means that if Y
follows the Poisson distribution, then the larger the mean of Y , the larger
its variance. (In (4.35), the notation k!, pronounced “k factorial”, is defined
as k! = k × (k − 1) × (k − 2) × . . . × 3 × 2 × 1.)

The Poisson distribution is typically used to model counts; this is a natural
choice for a number of reasons, including the fact that counts, like
the Poisson distribution, take on nonnegative integer values. To see how
we might use the Poisson distribution in practice, let Y denote the number
of users of the bike sharing program during a particular hour of the
day, under a particular set of weather conditions, and during a particular
month of the year. We might model Y as a Poisson distribution with
mean E(Y) = $\lambda$ = 5. This means that the probability of no users
during this particular hour is $\mathbb{P} (Y=0) = \frac{e^{-5} 5^0}{0!} = e^{-5} = 0.0067$ (where 0! = 1 by this convention).
The probabi9lity that there is exactly one user is $mathbb{P}(Y=1) = \frac{e^{-5}5^1}{1!} = 5e^{-5} = 0.034$, the probability of 
two users = 0.084, and so on.

Of course, in reality, we expect the mean number of users of the bike sharing program , $\lambda = E(Y)$, to vary as a function of the hour of the day, the monthg of the year, the weather conditions, and so forth. So rather
than modeling the number of bikers, Y , as a Poisson distribution with a
fixed mean value like $\lambda = 5$, we would like to allow the mean to vary as a
function of the covariates. In particular, we consider the following model
for the mean $\lambda = E(Y)$, which we now write as $\lambda(X_1, . . . ,X_p)$ to emphasize
that it is a function of the covariates:

$$log(\lambda(X_1, ... X_p)) = \beta_0 + \beta_1 X_1 + ... + \beta_p X_p \tag{4.36}$$

or equivalently

$$\lambda(X_1, ... , X_p) = e^{\beta_0 + \beta_1 X_1 + ... + \beta_p X_p} \tag{4.37}$$

Here, $\beta_0, \beta_1, ... , \beta_p$ are parameters to be estimated. Together, (4.35) and (4.36) define the Poisson regression model.
Notice that we take the *log* of $\lambda(X_1, ... , X_p)$ to be linear in $X_1, ... , X_p$ rather than having $\lambda(X_1, ... , X_p)$ itself be linear: This ensures that all values of $\lambda(X_1, ... , X_p)$ are non-negative values for all values of the covariates.

To estimate the coefficientes $\beta_0, \beta_1, ... , \beta_p$, we use the same maximum likelihood approach that we adopted for logistic regression in Section 4.3.2. Specifically, given *n* independent observations from the Poisson regression model, the likelihood takes the form:

$$\ell(\beta_0, \beta_1, \dots, \beta_p) = \prod_{i=1}^n \frac{e^{-\lambda(x_i)}\lambda(x_i)^{y_i}}{y_i!}, \tag{4.38}$$

where $\lambda(x_i) = e^{\beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip}}$, due to (4.37). We estimate the coefficients that maximize the likelihood $\ell(\beta_0, \beta_1, \dots, \beta_p)$, i.e. that make the observed data as likely as possible.

We now fit a Poisson regression model to the Bikeshare data set. The results are shown in Table 4.11 and Figure 4.15.

<img src="images for notes/Chapter4_tab4_11.png" width="600">

<img src="images for notes/Chapter4_fig4_15.png" width="600">

We again see that bike usage is highest in the spring and fall and during rush hour,
and lowest during the winter and in the early morning hours. Moreover,
bike usage increases as the temperature increases, and decreases as the
weather worsens. Interestingly, the coefficient associated with workingday
is statistically significant under the Poisson regression model, but not under
the linear regression model.
Some important distinctions between the Poisson regression model and
the linear regression model are as follows:

• Interpretation: To interpret the coefficients in the Poisson regression
model, we must pay close attention to (4.37), which states that an
increase in Xj by one unit is associated with a change in E(Y) = $\lambda$
by a factor of exp($\beta_j$). For example, a change in weather from clear
to cloudy skies is associated with a change in mean bike usage by a
factor of exp(−0.08) = 0.923, i.e. on average, only 92.3% as many
people will use bikes when it is cloudy relative to when it is clear.
If the weather worsens further and it begins to rain, then the mean
bike usage will further change by a factor of exp(−0.5) = 0.607, i.e.
on average only 60.7% as many people will use bikes when it is rainy
relative to when it is cloudy.

• Mean-Variance relationship: $\lambda = E(Y) = Var(Y)$.

• Nonnegative fitted values: There are no negative predictions using the
Poisson regression model.

#### 4.6.3 Generalized Linear Models in Greater Generality.

We have now discussed three types of regression models: Linear, logistic and Poisson. These approaches share some common characteristics:

1. Each approach uses predictors $X_1, ... , X_p$ to predict a response $Y$. We assume that, conditional on $X_1, ... , X_p$, $Y$ belongs to a certain family of distributions. For linear regression, we typically assume that $Y$ follows a Gaussian or normal distribution. For logistic regression, we assume that $Y$ follows a Bernoulli distribution. Finally, for Poisson regression, we assume that $Y$ follows a Poisson Distribution.

2. Each approach models the mean of $Y$ as a function of the predictos. In linear regression, the mean of $Y$ takes the form:

   In linear regression, the mean of $Y$ takes the form

$$E(Y|X_1, \dots, X_p) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p, \tag{4.39}$$

i.e. it is a linear function of the predictors. For logistic regression, the mean instead takes the form

$$\begin{aligned}
E(Y|X_1, \dots, X_p) &= \text{Pr}(Y = 1|X_1, \dots, X_p) \\
&= \frac{e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}}{1 + e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}},
\end{aligned} \tag{4.40}$$

while for Poisson regression it takes the form

$$E(Y|X_1, \dots, X_p) = \lambda(X_1, \dots, X_p) = e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}. \tag{4.41}$$

## Notes for chapter 4:


"Logistic Function" := $ p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}} \tag{4.2} $

"Odds":= $\frac{p(X)}{1-p(X)}$

"Log Odds" (also known as *logit*) := $ log \left( \frac{p(X)}{1-p(X)} \right)$

"Confounding": A bias in the estimation of the effect of a predictor variable on a response variable, due to an (or various?) unknown predictor variables.

So to start:

$p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1+e^{\beta_0 + \beta_1 X}}$

thus:

$ln(p(X)) = ln\left( \frac{e^{\beta_0 + \beta_1 X}}{1+e^{\beta_0 + \beta_1 X}}  \right)$

$\hspace{41pt} = ln\left( e^{\beta_0 + \beta_1 X} \right) - ln \left( 1+e^{\beta_0 + \beta_1 X} \right)$

$\hspace{41pt} = \beta_0 + \beta_1 X - ln \left( 1+e^{\beta_0 + \beta_1 X} \right)$



Positive Class = Thing we care about "catching", e.g. patient has cancer in cancer test.

**Sensitivity**: The proportion of *actual* positivies that were correctly identified by the model:

$\hspace{20pt} \frac{\text{True Positives}}{\text{Actual Positives}} = \frac{TP}{TP + FN}$

**Specificity**: The proportion of *actual* negatives that were correctly identified by the model.

$\text{Specificity} = \frac{\text{True Negatives}}{\text{Actual Negatives}} = \frac{TN}{TN + FP}$

**ROC Curve**: A receiver operating characteristic curve, or ROC curve, is a graphical plot that illustrates the performance of a binary classifier model (although it can be generalized to multiple classes) at varying threshold values. ROC analysis is commonly applied in the assessment of diagnostic test performance in clinical epidemiology. 

**What we are maximizing**: $$\ell(\beta) = \sum_{i=1}^{n} \sum_{k=1}^{K} y_{ik} \log(p_{ik})$$

In [5]:
import numpy as np

x = [0,1,2,3,4,5,6]

